![](https://www.pata.org/wp-content/uploads/2014/09/TripAdvisor_Logo-300x119.png)
# Predict TripAdvisor Rating
## ? ???? ???????????? ??? ????????? ??????????? ??????? ????????? ? TripAdvisor
**?? ???? ??????:**
* ????????? ?????? ? pandas
* ???????? ???????? ? Kaggle Notebooks
* ?????? ??? ?????? ????????????? ????????? ??????
* ???????? ???????? ? ???????????? ??????? (Nan)
* ???????????? ? ?????????? ?????? ??????????? ?????????
* ??????? ????????? [Feature Engineering](https://ru.wikipedia.org/wiki/???????????????_?????????) (???????????? ????? ????????)
* ? ?????? ??????? ???????? ML
* ? ?????? ??????...   



### ? ????? ??????, ??? ??? ?? ??????? ??????? ??????????????!

*???? ??????? ????????? ????????/???????? ? ????? ???????????? (Baseline) ? ?? ?????? ??????? ????????!*   
?? ?????? ???????????? ??? ??? ?????? ??? ?????????? ?????? ???????.

> ??? ????? baseline ???????, ????? ??? ????? ? ?????? ????????????? baseline ? ???????????? ????? ?????? ?????????? ?? kaggle ? ?????? ?????????.   
**baseline** ????????? ?????? ??? ??????, ??? ????? ?????????? ??? ?????????? ????????? ? ????????? ??????? ? ??? ????? ???????? ?? ??????. ??? ???? ?? ??????? ????? ???? ?????????? ???????, ?????? ??? ???????. ??? ???????? ??????? ?????????? ? ?????? ??, ? ?? ??????? ?????? ????? ?? ????? ?????????? ??????. 
????? baseline ????????? ??????? ??????? ?????? ?? ???????. ???? ???? ??????? ???? baseline - ?? ???? ??????? ???-?? ?? ?? ? ????? ??????????? ?????? ????) 

? ????????? ?????? ???????????? baseline ???? ? ?????????? ????????? ????, ??? ????? ?????? ? ???????, ? ? ???????????, ??? ?????? ??????, ????? ???????? ?????????.  ?????? ??????? ???????? ??? ?????? ???????, ??? ??? ???????????? ????? 2 ????? ??????? ???????? (? ????????? ???????????).

# import

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline

# ????????? ??????????? ??????? ?????????? ??? ?????????? ????????:
from sklearn.model_selection import train_test_split

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

In [ ]:
# ?????? ?????????? RANDOM_SEED, ????? ???? ???????????? ???? ??????????????!
RANDOM_SEED = 42

In [ ]:
# ??????????? ?????? ???????, ????? ???????????? ???? ??????????????:
!pip freeze > requirements.txt

# DATA

In [ ]:
DATA_DIR = '/kaggle/input/sf-dst-restaurant-rating/'
df_train = pd.read_csv(DATA_DIR+'/main_task.csv')
df_test = pd.read_csv(DATA_DIR+'kaggle_task.csv')
sample_submission = pd.read_csv(DATA_DIR+'/sample_submission.csv')

In [ ]:
df_train.info()

In [ ]:
df_train.head(5)

In [ ]:
df_test.info()

In [ ]:
df_test.head(5)

In [ ]:
sample_submission.head(5)

In [ ]:
sample_submission.info()

In [ ]:
# ?????! ??? ?????????? ????????? ????????? ?????????? ????? ? ???? ? ???? ???????
df_train['sample'] = 1 # ???????? ??? ? ??? ?????
df_test['sample'] = 0 # ???????? ??? ? ??? ????
df_test['Rating'] = 0 # ? ????? ? ??? ??? ???????? Rating, ?? ??? ?????? ???????????, ?? ????? ???? ?????? ????????? ??????

data = df_test.append(df_train, sort=False).reset_index(drop=True) # ??????????

In [ ]:
data.info()

????????? ?? ?????????:
* City: ????? 
* Cuisine Style: ?????
* Ranking: ???? ????????? ???????????? ?????? ?????????? ? ???? ??????
* Price Range: ???? ? ????????? ? 3 ??????????
* Number of Reviews: ?????????? ???????
* Reviews: 2 ????????? ?????? ? ???? ???? ???????
* URL_TA: ???????? ????????? ?? 'www.tripadvisor.com' 
* ID_TA: ID ????????? ? TripAdvisor
* Rating: ??????? ?????????

In [ ]:
data.sample(5)

In [ ]:
data.Reviews[1]

??? ?????, ??????????? ????????? ? ??? ??????? ??????? ? ??????????????? ?????????.

# Cleaning and Prepping Data
?????? ?????? ???????? ? ???? ???? ??????, ??????? ?????????? ?????????, ??? ???? ????? ???????? ?? ? ?????????? ??????. ?????? ?????? — ??? ??????????? ???? ??????? ????? ????? ???????? ??????.   
![](https://analyticsindiamag.com/wp-content/uploads/2018/01/data-cleaning.png)

## 1. ????????? NAN 
? ??????? ????????? ????? ???? ?????? ???????, ?? ???????? ????? ???? ?????????, ???? ????????? ?? ?????? ?????????. ?? ? ?????????? ????? ???? ????????????, **???? ?????????? ?????????? ????? ???? ?????? ?????????!**   
?? ????? ????? ?????????? NAN ????? ??????? ?????????? ? ??????? ???????? ??? ????????? ??????? 

In [ ]:
# ??? ??????? ? ?????? ??????? Number of Reviews
data['Number_of_Reviews_isNAN'] = pd.isna(data['Number of Reviews']).astype('uint8')

In [ ]:
data['Number_of_Reviews_isNAN']

In [ ]:
# ????? ????????? ???????? 0, ?? ?????? ??????????? ??????????? ??????? ??? ??????? ?? ?????? ? ??...
data['Number of Reviews'].fillna(0, inplace=True)

### 2. ????????? ?????????
??? ?????? ????????? ????? ???????? ? ??? ????? ???? ???????????????.

In [ ]:
data.nunique(dropna=False)

????? ???????? ????? ??????? ????????????????

??? ??????????? ?????????????? ????????? ???? ????????? ????????:
* Label Encoding
* One-Hot Encoding
* Target Encoding
* Hashing

????? ??????????? ??????? ?? ???????? ? ???????? ??????.
?? ????? ?????? ?????? ??????????? ? ??? ????????, ??????? ????????? ????? ?????? ? One-Hot Encoding:
![](https://i.imgur.com/mtimFxh.png)

In [ ]:
# ??? One-Hot Encoding ? pandas ???? ??????? ??????? - get_dummies. ???????? ?????? ???????? dummy_na
data = pd.get_dummies(data, columns=[ 'City',], dummy_na=True)

In [ ]:
data.head(5)

In [ ]:
data.sample(5)

#### ??????? ????????? ??????? "Price Range".

In [ ]:
data['Price Range'].value_counts()

?? ???????? 'Price Range' ??? - ???? ? ?????????.  
?? ????? ????????? ?? ??????????? (?????? ??? ?? ?????????????? ???????). ? ??? ??????, ??? ?? ????? ???????? ????????????????? ???????, ???????? 1,2,3  
*?????????? ??????? ????????? ????? ???????? ??? ??????????????!*

In [ ]:
# ???? ????????? 'Price Range'

> ??? ????????? ?????????? ?? ???? ??? ?? ?????????????? ????????? ????? ????????? One-Hot Encoding, ? ??? ????? ???????? ???????? ??????. ???????? ?????? ??????? ? ??????????? ???????? - ????? ?? ????? ???????, ??? ????? ????????.

### ?????????? ?????? ???????? ?? ?????? ??????????????!
??? ????????? ?????? ????????? ??? ???????? ???????? ???? ???????? ???? ???????, ? ????? ???? ? ?? ????, ?? ? ???? ? ???? ???? ???????? ? ???? ??????!     
?????? ?????????? ? ?????? ?? ??????? ????? ???????? ??????, ??? ??????? ??? ??????????????.

In [ ]:
# ??? ??? ??? ?? ????????? ?????? ?????????
# .....

![](https://cs10.pikabu.ru/post_img/2018/09/06/11/1536261023140110012.jpg)

# EDA 
[Exploratory Data Analysis](https://ru.wikipedia.org/wiki/???????????_??????_??????) - ?????? ??????
?? ???? ????? ?? ?????? ???????, ???? ??????????????, ????????, ??????? ??? ????? ????? ??????????.
? ????? ???? ????? ????? ??????, ??? ??? ?????? ????? ??? ???? ? ??? ???????? ????? ???? ????????????? ????? ?????.
????????? ??????????? ????????? ???????? ????????????? ?????, ????? ??????? ?, ??? ?????, ??????? ???? ?????? ?????.
![](https://miro.medium.com/max/2598/1*RXdMb7Uk6mGqWqPguHULaQ.png)

### ????????? ????????????? ????????

In [ ]:
plt.rcParams['figure.figsize'] = (10,7)
df_train['Ranking'].hist(bins=100)

? ??? ????? ??????????, ??????? ?? ?????????? ? ?? 2500 ????? ? ????? ??????, ? ??? ??? ?? ????????

In [ ]:
df_train['City'].value_counts(ascending=True).plot(kind='barh')

? ???-?? ???????, ??? ???????? ????? ??????=) ?????????, ??? ????????? ????????????? ? ??????? ??????:

In [ ]:
df_train['Ranking'][df_train['City'] =='London'].hist(bins=100)

In [ ]:
# ????????? ?? ??? 10 ???????
for x in (df_train['City'].value_counts())[0:10].index:
    df_train['Ranking'][df_train['City'] == x].hist(bins=100)
plt.show()

??????????, ??? Ranking ????? ?????????? ?????????????, ?????? ? ??????? ??????? ?????? ??????????, ??-?? ?? ????? ????? ????????.

>????????? ??? ?? ????? ????? ??????? ??????? ??? ????? ??????. ? ?????? ??? ??????, ??? ???????????? ???????? ???????? ???????????. ? ????? ?????????? ??? ????????? =) 


### ????????? ????????????? ??????? ??????????

In [ ]:
df_train['Rating'].value_counts(ascending=True).plot(kind='barh')

### ????????? ????????????? ??????? ?????????? ???????????? ????????

In [ ]:
df_train['Ranking'][df_train['Rating'] == 5].hist(bins=100)

In [ ]:
df_train['Ranking'][df_train['Rating'] < 4].hist(bins=100)

### ? ???? ?? ???? ??????? - [?????????? ?????????](https://ru.wikipedia.org/wiki/??????????)
?? ???? ??????? ??? ?????? ?? ??????? ????????, ??? ???????? ??????? ????? ????? ? ? ??????? ??????????.

In [ ]:
plt.rcParams['figure.figsize'] = (15,10)
sns.heatmap(data.drop(['sample'], axis=1).corr(),)

?????? ????????? ???????????? ? ???? ???????? ????? ?????? ????? ?????????? ??????, ????????:
* ??? ?????? ??????? ? ??????? ??? ????????
* ? ????? ?????? ????? ?????????? ????? ?????????????

?????????? ???? ?????? ? ??????? ?? ???? ????? ? ??????)

# Data Preprocessing
??????, ??? ???????? ? ????????????????? ????, ???????? ??? ????????? ? ???? ??????? ???????.

In [ ]:
# ?? ?????? ??????, ?????? ?????????? ??????
df_train = pd.read_csv(DATA_DIR+'/main_task.csv')
df_test = pd.read_csv(DATA_DIR+'/kaggle_task.csv')
df_train['sample'] = 1 # ???????? ??? ? ??? ?????
df_test['sample'] = 0 # ???????? ??? ? ??? ????
df_test['Rating'] = 0 # ? ????? ? ??? ??? ???????? Rating, ?? ??? ?????? ???????????, ?? ????? ???? ?????? ????????? ??????

data = df_test.append(df_train, sort=False).reset_index(drop=True) # ??????????
data.info()

In [ ]:
def preproc_data(df_input):
    '''includes several functions to pre-process the predictor data.'''
    
    df_output = df_input.copy()
    
    # ################### 1. ????????????? ############################################################## 
    # ??????? ?? ?????? ??? ?????? ????????
    df_output.drop(['Restaurant_id','ID_TA',], axis = 1, inplace=True)
    
    
    # ################### 2. NAN ############################################################## 
    # ????? ????????? ????????, ?? ?????? ??????????? ??????????? ??????? ??? ??????? ?? ?????? ? ??...
    df_output['Number of Reviews'].fillna(0, inplace=True)
    # ??? ??? ??? ?? ????????? NAN
    # ....
    
    
    # ################### 3. Encoding ############################################################## 
    # ??? One-Hot Encoding ? pandas ???? ??????? ??????? - get_dummies. ???????? ?????? ???????? dummy_na
    df_output = pd.get_dummies(df_output, columns=[ 'City',], dummy_na=True)
    # ??? ??? ??? ?? Encoding ??????
    # ....
    
    
    # ################### 4. Feature Engineering ####################################################
    # ??? ??? ??? ?? ????????? ????? ??????
    # ....
    
    
    # ################### 5. Clean #################################################### 
    # ??????? ???????? ??????? ??? ?? ?????? ??????????, 
    # ?????? ?? ????????? ? dtypes "object" ????????? ?? ?????, ?????? ??????? ?? ? ??????
    object_columns = [s for s in df_output.columns if df_output[s].dtypes == 'object']
    df_output.drop(object_columns, axis = 1, inplace=True)
    
    return df_output

>?? ????????, ????? ???? ?? ????????? ??? ??????? ??????? ? ????? ? ??????? ?? ?????????? (???????? ???). 

#### ????????? ? ????????? ??? ??????????

In [ ]:
df_preproc = preproc_data(data)
df_preproc.sample(10)

In [ ]:
df_preproc.info()

In [ ]:
# ?????? ??????? ???????? ?????
train_data = df_preproc.query('sample == 1').drop(['sample'], axis=1)
test_data = df_preproc.query('sample == 0').drop(['sample'], axis=1)

y = train_data.Rating.values            # ??? ??????
X = train_data.drop(['Rating'], axis=1)

**????? ??? ??? ?????????? ???? ?????? ?? ????????, ???????? ?????? ?? ??? ???? ???? ? ?????, ??? ?????????. 
??? ??????? ??? ?????????, ??? ?????? ???? ?????? ????????, ?? ???????? submissiona ?? kaggle.**

In [ ]:
# ????????????? ??????????? ??????? train_test_split ??? ???????? ???????? ??????
# ??????? 20% ?????? ?? ????????? (???????? test_size)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
# ?????????
test_data.shape, train_data.shape, X.shape, X_train.shape, X_test.shape

# Model 
??? ML

In [ ]:
# ??????????? ??????????? ??????????:
from sklearn.ensemble import RandomForestRegressor # ?????????? ??? ???????? ? ???????? ??????
from sklearn import metrics # ??????????? ??? ?????? ???????? ??????

In [ ]:
# ??????? ?????? (????????? ?? ???????)
model = RandomForestRegressor(n_estimators=100, verbose=1, n_jobs=-1, random_state=RANDOM_SEED)

In [ ]:
# ??????? ?????? ?? ???????? ?????? ??????
model.fit(X_train, y_train)

# ?????????? ????????? ?????? ??? ???????????? ???????? ?????????? ? ???????? ???????.
# ????????????? ???????? ?????????? ? ?????????? y_pred
y_pred = model.predict(X_test)

In [ ]:
# ?????????? ????????????? ???????? (y_pred) ? ????????? (y_test), ? ??????? ????????? ??? ? ??????? ??????????
# ??????? ?????????? Mean Absolute Error (MAE) ? ?????????? ??????? ?????????? ????????????? ???????? ?? ???????????.
print('MAE:', metrics.mean_absolute_error(y_test, y_pred))

In [ ]:
# ? RandomForestRegressor ???? ??????????? ??????? ????? ?????? ???????? ??? ??????
plt.rcParams['figure.figsize'] = (10,10)
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
feat_importances.nlargest(15).plot(kind='barh')

# Submission
???? ??? ?????????? - ??????? Submission ?? ????

In [ ]:
test_data.sample(10)

In [ ]:
test_data = test_data.drop(['Rating'], axis=1)

In [ ]:
sample_submission

In [ ]:
predict_submission = model.predict(test_data)

In [ ]:
predict_submission

In [ ]:
sample_submission['Rating'] = predict_submission
sample_submission.to_csv('submission.csv', index=False)
sample_submission.head(10)

# What's next?
??? ??? ??????, ???? ???????? ?????????:
* ?????????? ?????????? ???????? ? ???????? ??? ?????? ??????
* ??????????, ??? ??? ????? ??????? ?? ?????????
* ????????????? ????? ????????
* ?????????? ?????????????? ??????, ????????: ?? ????????? ??? ?????????????? ???????
* ????????? ?????? ?????????

? ?????, ??????? ?????????? ? ?????? ?????????????! ????? ? ????????????!
